In [105]:
from anthropic import Anthropic
import json
import pprint

client = Anthropic()
model = "claude-haiku-4-5"

In [106]:
messages = []

In [107]:
# Helper functions to manage messages

def add_user_message(messages, user_message):
    messages.append({"role": "user", "content": user_message})


def add_assistant_message(messages, assistant_message):
    messages.append({"role": "assistant", "content": assistant_message})


def chat(messages, stop_sequences=[]):

    response = client.messages.create(
        model = model,
        max_tokens = 1000,
        messages = messages,
        temperature = 1.0,
        stop_sequences = stop_sequences
    )

    return response.content[0].text

In [108]:
def run_prompt(user_description):
    prompt = f"""
Format the start time, end time, and the work that was done as JSON from the below user prompt.
{user_description}

Example output:
```json
[
    {{
        "task": "coding",
        "start_time": "12:00",
        "end_time": "14:00"
    }},
    ...additional
]```
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences=["```"])
    return json.loads(response.strip())


In [109]:
def validate_by_model(user_description, output):
    eval_prompt = f"""
You are an expert in reviewing Python dictionaries that were created from
natural language descriptions.
Your task is to review the dictionary output generated by a model based on
the natural language description of various tasks done in different times of
the same day.

<user_description>
{user_description}
</user_description>

JSON to Evaluate:
<output>
{output}
</output>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "is_valid": True if the JSON output is a correct and complete representation of the description, otherwise False

Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "is_valid": boolean
}}
"""
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [110]:
def validate_by_code(output):
    # Implement your code-based validation logic here
    # For example, you could check if the JSON output has the expected structure,
    # if the time formats are correct, etc.
    try:
        for entry in output:
            if "task" not in entry or "start_time" not in entry or "end_time" not in entry:
                return False
        return True
    except Exception as e:
        print(f"Validation error: {e}")
        return False

In [111]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""

    user_description = test_case["description"]

    output = run_prompt(user_description)

    result_from_model = validate_by_model(user_description, output)
    is_validated_by_code = validate_by_code(output)

    return {
        "user_description": user_description,
        "output": output,
        "reasoning": result_from_model["reasoning"],
        "is_valid": result_from_model["is_valid"] and is_validated_by_code
    }

In [112]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [113]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation.
The dataset will be used to evaluate prompts that generate JSON outputs.
The description should contain tasks done at different times of the same day.
Make sure the start and end time for each task is included in the description.
Keep the description concise and limited to 100 characters.

Example output:
```json
[
    {
        "description": "From noon, I did a bit of coding for two hours. Then I had a break for an hour. After that, I spent 30 minutes reading a book.",
    },
    ...additional
]
```
Please generate 3 object.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [114]:
dataset = generate_dataset()
results = run_eval(dataset)


In [ ]:
print(f"Number of Tests: {len(results)}")
score = sum(result["is_valid"] for result in results) / float(len(results))
print(f"Score: {score} / 1.0")
print("Results:")
pprint.pprint(results)
